In [ ]:
# Project 2: Travel, Tourism & Hospitality Analysis
## Week 1: Data Acquisition, Cleaning, and Feature Engineering

### Executive Summary
The primary objective of this notebook is to process historical hospitality booking records to prepare the data for downstream
Exploratory Data Analysis (EDA) and predictive churn modeling. This includes rigorous data cleaning,missing value imputation, 
extreme outlier handling, and creating composite features.


In [9]:
import pandas as pd
import numpy as np

# Load the raw dataset
file_path = 'hotel_bookings.csv'
df = pd.read_csv(file_path)

print(f"Initial raw dataset footprint:")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()


Initial raw dataset footprint:
Rows: 119390 | Columns: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [ ]:
### Day 2: Handling Null Values in Critical Columns
The `company` and `agent` IDs contain numerous missing configurations. In hotel booking systems, a blank agency/company field logically 
defaults to zero (representing a direct or independent reservation). We will also patch secondary columns.

In [10]:
# Impute critical columns to reflect "0" (Direct Booking)
if 'company' in df.columns:
    df['company'] = df['company'].fillna(0)
if 'agent' in df.columns:
    df['agent'] = df['agent'].fillna(0)

# Patch secondary demographic and booking traits
if 'children' in df.columns:
    df['children'] = df['children'].fillna(0)
if 'country' in df.columns:
    df['country'] = df['country'].fillna('Unknown')

print("Missing values standardized successfully.")


Missing values standardized successfully.


In [ ]:
## Day 3: Scrubbing ADR (Average Daily Rate) Outliers
To ensure our downstream Dynamic Pricing and Churn models operate on accurate financial data, we must remove impossible price entries.


In [11]:
if 'adr' in df.columns:
    # Identify impossible pricing anomalies
    outlier_condition = (df['adr'] < 0) | (df['adr'] > 4000)
    anomalies_found = df[outlier_condition].shape[0]
    
 # Secure the dataset by filtering out the anomalies
    df = df[~outlier_condition]
    print(f"Successfully removed {anomalies_found} outlier(s) from the ADR financial column.")


Successfully removed 2 outlier(s) from the ADR financial column.


In [ ]:
## Day 4: Feature Engineering
A foundational predictor for cancellations is the total length of the reservation. We need to calculate the `total_duration_of_stay`
and drop erroneous zero-day bookings.


In [12]:
if 'stays_in_weekend_nights' in df.columns and 'stays_in_week_nights' in df.columns:
    # Combine weekend and week nights into a single consolidated duration metric
    df['total_duration_of_stay'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
    
 # Identify and drop ghosts/glitches where a user stayed for 0 days
zero_stay_mask = (df['total_duration_of_stay'] == 0)
zero_stays_found = zero_stay_mask.sum()
df = df[~zero_stay_mask]
    
print(f"Dropped {zero_stays_found} zero-day bookings. Feature created successfully.")


Dropped 715 zero-day bookings. Feature created successfully.


In [ ]:
## Day 5: Memory Optimization & Data Type Corrections
To ensure our downstream algorithms run efficiently, we must optimize the dataset's memory footprint by downcasting incorrect data types.


In [13]:
# Convert the target cancellation column from a bulky integer to an efficient boolean
if 'is_canceled' in df.columns:
    df['is_canceled'] = df['is_canceled'].astype(bool)
# Convert demographic floats (like 1.0 children) to strict integers
if 'children' in df.columns:
    df['children'] = df['children'].astype(int)
    
print("Day 6: Downcasted data types and system memory optimized successfully.")


Day 6: Downcasted data types and system memory optimized successfully.
